In [0]:
%run ../01_Bronze/00_setup

In [0]:
# process new records into the Silver layer robots table
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, current_date
from pyspark.sql import Window


def process_scd2_robots(microBatchDF, batchId):
    # 1. Get a list of order_ids that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.robots"
    
    window_spec = Window.partitionBy("robot_id").orderBy(F.col("event_timestamp").desc())
    
    # This keeps only the most recent status for each robot in the current micro-batch
    deduped_df = microBatchDF \
        .withColumn("rn", F.row_number().over(window_spec)) \
        .filter("rn = 1") \
        .drop("rn")
    
    # 2. Split the micro-batch logic using the DEDUPED data
    # Part A: The 'Insert' half
    inserts_df = deduped_df.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half
    existing_ids = deduped_df.sparkSession.table(target_table) \
        .filter("is_active = true") \
        .select("robot_id")
    
    updates_df = deduped_df.join(existing_ids, "robot_id", "inner") \
        .withColumn("merge_key", F.col("robot_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    
    # Register the view from the combined and cleaned dataframe
    combined_df.createOrReplaceTempView("source_robot_view")

  
    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_robot_view AS source
        ON target.robot_id = source.merge_key AND target.is_active = true
        
        -- If we find the ID and status changed, expire the old one
        WHEN MATCHED AND target.status <> source.status THEN
          UPDATE SET target.is_active = false, target.end_date = current_timestamp(), target.updated_timestamp = current_timestamp()
          
        -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
        WHEN NOT MATCHED THEN
          INSERT (robot_sk, robot_id, robot_type, max_weight_capacity, max_speed_mps, status, current_battery_pct, last_maintenance_date,  is_active, start_date, end_date, updated_timestamp)
          VALUES (md5(concat(source.robot_id, cast(source.event_timestamp as STRING))), 
                  source.robot_id, 'Hercules', source.max_weight_capacity, 10.0, source.status, source.battery_level, current_date(), 
                  true, source.event_timestamp, NULL, current_timestamp())
    """)

    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.robot_events")
    .filter("status IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd2_robots)
    .option("checkpointLocation", f"{silver_volume_path}/checkpoints/silver_robots_scd2")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())

In [0]:
%sql
describe extended workspace.amazon_fullfilment_bronze.robot_events

In [0]:
%sql
select * from workspace.amazon_fullfilment_bronze.robot_events where robot_id='ROB_001'

In [0]:
%sql
select * from workspace.amazon_fullfilment_bronze.robot_events where robot_id='ROB_603'

In [0]:
%sql
select * from workspace.amazon_fullfilment_silver.robots where robot_id='ROB_603'

In [0]:
%sql
describe history workspace.amazon_fullfilment_bronze.robot_events

In [0]:
%sql
select * from workspace.amazon_fullfilment_silver.robots